## Indexing

### kid-friend

In [1]:
import json
import pandas as pd


lines = []
with open(r'../data/kid-friend-en/en/inputs/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
kid_friend_df_final = pd.DataFrame(line_dicts)
kid_friend_df_final.drop_duplicates(inplace=True,ignore_index=True)
kid_friend_df_final.head(1)

,docno,snippet,title,main_content
0,6e421f1539b1457b853712d81be87743,WEBBTS (also Bangtan Boys; Korean: 방탄소년단 Bangt...,BTS (band) - Wikipedia,BTS (also Bangtan Boys; Korean: 방탄소년단 Bangtan ...


In [2]:
print("Number of documents: ", len(kid_friend_df_final))
print("Longest document length (in characters): ", max([len(text) for text in kid_friend_df_final["main_content"]]))
print("Longest length of document title (in characters): ", max([len(title) for title in kid_friend_df_final["title"]]))
print("Maximum length of doc id (in characters): ", max([len(docno) for docno in kid_friend_df_final["docno"]]))

Number of documents:  2385
Longest document length (in characters):  117554
Longest length of document title (in characters):  452
Maximum length of doc id (in characters):  32


In [4]:
from pathlib import Path
import pyterrier as pt
import pandas as pd

kid_friend_df_final.to_dict(orient="records")

kid_friend_idx_path = "../data/kid-friend-en/index"
kid_friend_indexer = pt.IterDictIndexer(
    index_path = str(kid_friend_idx_path),
    meta={ # metadata recorded in index
        "docno": max([len(docno) for docno in kid_friend_df_final["docno"]]),
        "title": max([len(title) for title in kid_friend_df_final["title"]]),
        "snippet": max([len(snippet) for snippet in kid_friend_df_final["snippet"]]),
        "text": max([len(main_content) for main_content in kid_friend_df_final["main_content"]])
    },
    text_attrs = ["text"], # columns indexed
    stemmer="porter",
    stopwords="terrier",
)

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [5]:
def kid_friend_iter():
    for i, row in kid_friend_df_final.iterrows():
        yield {
            "docno": row["docno"],
            "title": row["title"],
            "snippet": row["snippet"],
            "text": row["main_content"]
        }

In [6]:
kid_friend_index_ref = kid_friend_indexer.index(kid_friend_iter())

11:09:49.545 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (37d53c33110a4231807e71e377f08377) - further warnings are suppressed
11:09:53.366 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 187 empty documents


In [7]:
len(kid_friend_df_final.loc[kid_friend_df_final["main_content"]==""])

187

### requik

In [9]:
lines = []
with open(r'../data/requik/corpus/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
requik_df_final = pd.DataFrame(line_dicts)
requik_df_final = requik_df_final.rename({"id":"docno", "contents": "main_content"}, axis=1)
requik_df_final.drop_duplicates(inplace=True,ignore_index=True)
requik_df_final.head(1)

,docno,main_content
0,RQ_d1,When was the last time the capulin volcano nat...


In [10]:
print("Number of documents: ", len(requik_df_final))
print("Longest document length (in characters): ", max([len(text) for text in requik_df_final["main_content"]]))
print("Maximum length of doc id (in characters): ", max([len(docno) for docno in requik_df_final["docno"]]))

Number of documents:  317
Longest document length (in characters):  1281
Maximum length of doc id (in characters):  7


In [13]:
requik_df_final.to_dict(orient="records")

requik_idx_path = "../data/requik/index"
requik_indexer = pt.IterDictIndexer(
    index_path = str(requik_idx_path),
    meta={ # metadata recorded in index
        "docno": max([len(docno) for docno in requik_df_final["docno"]]),
        "text": max([len(main_content) for main_content in requik_df_final["main_content"]])
    },
    text_attrs = ["text"], # columns indexed
    stemmer="porter",
    stopwords="terrier",
)

In [14]:
def requik_iter():
    for i, row in requik_df_final.iterrows():
        yield {
            "docno": row["docno"],
            "text": row["main_content"]
        }

In [15]:
requik_index_ref = requik_indexer.index(requik_iter())

In [17]:
len(requik_df_final.loc[requik_df_final["main_content"]==""])

0

## Retrieval

In [1]:
from pathlib import Path
import pyterrier as pt
import pandas as pd

kid_friend_queries = pd.read_csv("../data/kid-friend-en/en/inputs/topics.csv")
kid_friend_queries["qid"] = kid_friend_queries["qid"].astype(str)
# kid_friend_topical_qrels = pd.read_csv("../data/kid-friend-en/en/qrels/qrels-relevance-binary.csv")
# kid_friend_topical_qrels["qid"] = kid_friend_topical_qrels["qid"].astype(str)

kid_friend_idx_path = "../data/kid-friend-en/index"
kid_friend_index = pt.IndexFactory.of(str(kid_friend_idx_path))

Java started (triggered by IndexFactory.of) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [2]:
requik_queries = pd.read_csv("../data/requik/inputs/topics.csv")
requik_queries["qid"] = requik_queries["qid"].astype(str)

requik_idx_path = "../data/requik/index"
requik_index = pt.IndexFactory.of(str(requik_idx_path))

In [3]:
from pyterrier_caching import RetrieverCache

## simple first stage retrievers
kid_friend_tf_idf = pt.terrier.Retriever(kid_friend_index, wmodel="TF_IDF")
kid_friend_bm25 = pt.terrier.Retriever(kid_friend_index, wmodel="BM25")
kid_friend_dirichlet_lm = pt.terrier.Retriever(kid_friend_index, wmodel="DirichletLM")

requik_tf_idf = pt.terrier.Retriever(requik_index, wmodel="TF_IDF")
requik_bm25 = pt.terrier.Retriever(requik_index, wmodel="BM25")
requik_dirichlet_lm = pt.terrier.Retriever(requik_index, wmodel="DirichletLM")

# caching the first 100 resources retrieved by bm25 for each dataset to be used for re-rankers
kid_friend_bm25_cached_retriever = RetrieverCache('./cache/kid-friend-en', kid_friend_bm25 % 100 >> pt.text.get_text(kid_friend_index, 'text'))
requik_bm25_cached_retriever = RetrieverCache('./cache/requik', requik_bm25 % 100 >> pt.text.get_text(requik_index, 'text'))

In [5]:
## neural re-rankers: monoT5

from pyterrier_t5 import MonoT5ReRanker

monoT5 = MonoT5ReRanker(text_field='text', model = 'castorini/monot5-base-msmarco')
kid_friend_monoT5 = (kid_friend_bm25_cached_retriever >> monoT5)
requik_monoT5 = (requik_bm25_cached_retriever >> monoT5)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [13]:
kid_friend_bm25.search(kid_friend_queries["query"][2])

,qid,docid,docno,rank,score,query
0,1,554,1034d422a9154fd09d1ef847809c4d88,0,24.403817,BTS merch
1,1,565,42f8dede7c1a4818a7d91ac2b4486094,1,23.978815,BTS merch
2,1,562,6a7d04e19d5c4c589749780c39dab51f,2,23.971948,BTS merch
3,1,29,f449e7fd662e4a5f8f46fe67a20f2a85,3,23.709059,BTS merch
4,1,23,63175e0a62614f568ba8bb88537dfd97,4,22.455583,BTS merch
...,...,...,...,...,...,...
85,1,1379,efba881a2b0b4364852252b4144ca8f5,85,5.963200,BTS merch
86,1,703,7383555fdeb740df9f7c61f76b6fc73f,86,4.722670,BTS merch
87,1,759,6d39234cd20d46989480ebbbfe0407b1,87,2.169711,BTS merch
88,1,683,9ffc2553678d4372a329524ea974eb6d,88,2.077047,BTS merch


In [14]:
kid_friend_monoT5.search(kid_friend_queries["query"][2])

monoT5: 100%|██████████| 23/23 [00:02<00:00, 11.11batches/s]


,qid,docid,docno,query,text,score,rank
16,1,1725,c51ba51158df477b8616c7bda055788e,BTS merch,BTS - Seoul-Mate\n\nBTS\n\nFilter\n 143 pro...,-0.001540,0
11,1,1728,ec742a5f53bf4c6db7aafdc361afbf61,BTS merch,Skip to content\n10% new customer discount - C...,-0.001887,1
15,1,24,06c8038c50714812ae1fb562865444a1,BTS merch,Skip to content\n10% new customer discount - C...,-0.002117,2
3,1,29,f449e7fd662e4a5f8f46fe67a20f2a85,BTS merch,BTS Merch Germany : Clothing and accessories\n...,-0.002126,3
0,1,554,1034d422a9154fd09d1ef847809c4d88,BTS merch,Previous\nNext\n\nBTS merch categories\n\nBTS ...,-0.004141,4
...,...,...,...,...,...,...,...
89,1,353,9d5f4572038d4855bc2e53f2b799b833,BTS merch,This article also exists as an audio file.\n\n...,-12.699072,85
56,1,1376,9cf39de35fe944e2b83d91ad03a30cf6,BTS merch,Skip to content\n\nBangkok/Siam\n\nUseful arti...,-12.955190,86
87,1,759,6d39234cd20d46989480ebbbfe0407b1,BTS merch,Jump to content\n\nGladiator II\n\nFrom Wikipe...,-13.358723,87
86,1,703,7383555fdeb740df9f7c61f76b6fc73f,BTS merch,List of Official Disney Princesses (And the On...,-13.463104,88


In [49]:
# cutoffs = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
# kid_friend_bm25 = pt.terrier.Retriever(kid_friend_index, wmodel="BM25")
# res = pt.Experiment(
#     [kid_friend_bm25 % cutoff >> pt.text.get_text(kid_friend_index, 'text') >> monoT5 for cutoff in cutoffs],
#     kid_friend_queries,
#     kid_friend_topical_qrels,
#     names=[f'c={cutoff}' for cutoff in cutoffs],
#     eval_metrics=["map", "recip_rank", "ndcg", "ndcg_cut.10", "mrt"]
# )
# res

In [50]:
# from matplotlib import pyplot as plt
# %matplotlib inline
# plt.plot(res['name'], res['ndcg_cut.10'], label='ndcg_cut_10')
# plt.plot(res['name'], res['recip_rank'], label='recip_rank')
# plt.ylabel('ndcg_cut_10 / recip_rank')
# plt.legend()
# plt.show()
# plt.clf()
# plt.plot(res['name'], res['mrt'])
# plt.ylabel('mrt')
# plt.show()

In [4]:
from rerank import LLMReRanker
from rerank import PointwiseReranker
from huggingface_hub import login
import json

with open(r'access_tokens.json') as f:
    tokens = json.load(f)

login(token=tokens['HF_access_token'])
### Note to self: run export 'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True' for each model first in a terminal, 
###               reset the kernel, and then run the model

## listwise rerankers
# rank_vicuna = LLMReRanker("castorini/rank_vicuna_7b_v1")
# kid_friend_vicuna = kid_friend_bm25_cached_retriever >> rank_vicuna
# requik_vicuna = requik_bm25_cached_retriever >> rank_vicuna

# rank_zephyr = LLMReRanker("castorini/rank_zephyr_7b_v1_full")
# kid_friend_zephyr = kid_friend_bm25_cached_retriever >> rank_zephyr
# requik_zephyr = requik_bm25_cached_retriever >> rank_zephyr

## pointwise reranker

rank_llama = PointwiseReranker('castorini/rankllama-v1-7b-lora-passage')
kid_friend_rank_llama = kid_friend_bm25_cached_retriever >> rank_llama
requik_rank_llama = requik_bm25_cached_retriever >> rank_llama

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-31 14:08:00.040861: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-31 14:08:00.092423: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-31 14:08:01.091794: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You

OutOfMemoryError: CUDA out of memory. Tried to allocate 172.00 MiB. GPU 0 has a total capacity of 22.06 GiB of which 133.44 MiB is free. Including non-PyTorch memory, this process has 21.92 GiB memory in use. Of the allocated memory 21.69 GiB is allocated by PyTorch, and 10.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [5]:
kid_friend_zephyr.search(kid_friend_queries["query"][2])

100%|██████████| 1/1 [00:43<00:00, 43.56s/window]


,qid,docid,docno,rank_orig,score_0,query,text,score,rank
0,1,554,1034d422a9154fd09d1ef847809c4d88,0,24.403817,BTS merch,Previous\nNext\n\nBTS merch categories\n\nBTS ...,1.000000,0
1,1,565,42f8dede7c1a4818a7d91ac2b4486094,1,23.978815,BTS merch,[MERCH] Artist-Made Collection by BTS\n\n\n3rd...,0.500000,1
2,1,562,6a7d04e19d5c4c589749780c39dab51f,2,23.971948,BTS merch,[MERCH] Official BTS-Themed Merch in the USA\n...,0.333333,2
3,1,29,f449e7fd662e4a5f8f46fe67a20f2a85,3,23.709059,BTS merch,BTS Merch Germany : Clothing and accessories\n...,0.250000,3
16,1,1725,c51ba51158df477b8616c7bda055788e,16,9.953507,BTS merch,BTS - Seoul-Mate\n\nBTS\n\nFilter\n 143 pro...,0.200000,4
...,...,...,...,...,...,...,...,...,...
85,1,1379,efba881a2b0b4364852252b4144ca8f5,85,5.963200,BTS merch,"Susanne, Tina's mother\n\nTina and Holger's mo...",0.011628,85
86,1,703,7383555fdeb740df9f7c61f76b6fc73f,86,4.722670,BTS merch,List of Official Disney Princesses (And the On...,0.011494,86
87,1,759,6d39234cd20d46989480ebbbfe0407b1,87,2.169711,BTS merch,Jump to content\n\nGladiator II\n\nFrom Wikipe...,0.011364,87
88,1,683,9ffc2553678d4372a329524ea974eb6d,88,2.077047,BTS merch,List of Disney Films\n\nDisney Films A-Z\n\nFr...,0.011236,88
